In [0]:
emp_filePath = 'dbfs:/FileStore/shared_uploads/iriscloudone@outlook.com/emp.csv'

In [0]:
empdf = spark.read.format('csv')\
                  .option('header','true') \
                  .option('inferSchema','true') \
                  .load(emp_filePath)

In [0]:
display(empdf)

### 1. Create Delta Tables

In [0]:
empdf.write.format('delta') \
           .save('/tmp/emp-delta-table')

### 2. Read Delta Tables

In [0]:
new_empdf = spark.read.format('delta') \
                  .load('/tmp/emp-delta-table')
display(new_empdf)

In [0]:
display(dbutils.fs.ls('/tmp/emp-delta-table'))

In [0]:
display(dbutils.fs.ls('dbfs:/tmp/emp-delta-table/_delta_log/'))

In [0]:
spark.catalog.listDatabases()

### 3. Updates to Delta Table

In [0]:
empdf.write.format('delta') \
           .mode('overwrite') \
           .save('/tmp/emp-delta-table')

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, "/tmp/emp-delta-table")

In [0]:
deltaTable.update(
  condition = expr("emp_name == 'Jane Smith'"),
  set = { "salary": "70000" }
  )


In [0]:
updated_empdf = spark.read.format('delta') \
                  .load('/tmp/emp-delta-table')
display(updated_empdf)

In [0]:
deltaTable.update(
  condition = expr("emp_name == 'Jane Smith'"),
  set = { "salary": "65000" }
  )


In [0]:
updated_empdf = spark.read.format('delta') \
                  .load('/tmp/emp-delta-table')
display(updated_empdf)

In [0]:
display(deltaTable.history())

### 4. Time Travel


In [0]:
emp_historyVersion0 = spark.read.format('delta') \
                  .option("versionAsOf", 0) \
                  .load('/tmp/emp-delta-table')

In [0]:
emp_historyVersion1 = spark.read.format('delta') \
                  .option("versionAsOf", 1) \
                  .load('/tmp/emp-delta-table')

In [0]:
display(emp_historyVersion0)

In [0]:
display(emp_historyVersion1)

In [0]:
emp_historyVersion2 = spark.read.format('delta') \
                  .option("versionAsOf", 2) \
                  .load('/tmp/emp-delta-table')

In [0]:
emp_historyVersion3 = spark.read.format('delta') \
                  .option("versionAsOf", 3) \
                  .load('/tmp/emp-delta-table')

In [0]:
display(emp_historyVersion2)

In [0]:
display(emp_historyVersion3)

In [0]:
emp_latestHistory = spark.read.format('delta') \
                  .option("versionAsOf", 1) \
                  .load('/tmp/emp-delta-table')

In [0]:
display(emp_latestHistory)

In [0]:
empdf.write.format('delta').mode('overwrite').saveAsTable('emp_tbl')
display(spark.sql('select * from emp_tbl'))